# 03.3 — Testing behavior with pytest

[Previous: 03.2](03.2-branches-and-collaboration.ipynb) · [Class index and setup](README.md) · [Next: 03.4](03.4-fixtures-and-parametrized-tests.ipynb)

**Purpose:** Turn the intended result into a repeatable check that exposes a real defect.

**In class:** 30 minutes. **Independent work:** 15 minutes, within the index's 210-minute budget. The separate course project is additional workload.

## Learning outcomes

- Write a small arrange/act/assert test for a contract.
- Distinguish an assertion failure from a collection/import error.
- Fix behavior in a scratch copy without weakening its expected result.

## Starting point

You know basic Python. The complete Report/Observation package, data, practice inputs, and references are supplied in this class folder. No previous notebook, earlier exercise answer, or model/API access is needed. In VS Code, select this class's `.venv`, then use **Restart Kernel → Run All**. Every part imports its own names.

Default cells read local examples and may run local checks; they do not mutate Git repositories, change configuration, install packages, read credentials, or contact GitHub. All Git writing commands below are manual terminal work in a NEW scratch copy outside the course checkout. Empty answer cells are safe. Attempt before uncommenting `%load` in its separate cell: run to reveal source, then run that populated cell separately. Loading may replace its invoking cell; a kernel restart does not remove loaded source.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_03_3_e1.py").is_file() or not Path("report_app/models.py").is_file():
    raise FileNotFoundError(
        "Open 03-git-collaboration-and-testing.code-workspace and restart the kernel. "
        "Download the complete folder, including report_app/, practice/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available.")

## In class · 60–80 minutes — Specify behavior before trusting a result

A test arranges inputs, acts on the code, and asserts an expected outcome. The expectation should come from a requirement or a known example, not by copying the current function's output. The small quantity-report contract is explicit: `[2, 4]` has mean `3`; empty input has mean `None`; `[0]` has mean zero. Missing measurements and measured zero are different facts.

pytest collects functions named `test_*` in test modules. Running `python -m pytest` from the project root uses that Python and makes normal package imports available. Open [the passing suite](tests/test_reports.py) to see how it imports the supplied package. No notebook import or hidden previous execution is involved.

In [ ]:
from report_app.models import Observation
from report_app.analysis import summarize

# Arrange: the expected result is calculated from a known small example.
records = [Observation(label="a", value=2), Observation(label="b", value=4)]
# Act:
actual = summarize(records)
# Assert:
assert actual == {"count": 2, "total": 6, "mean": 3}
assert summarize([])["mean"] is None
print(actual)

### Visual guide — Keep the requirement fixed while correcting code

<img src="images/behavior-test-cycle.png" alt="Known inputs lead to an actual result and an assertion against the requirement; an assertion failure leads to correcting code and rerunning the same check." width="784" style="max-width: 100%; height: auto; background-color: white;">

Arrange known inputs, call the function, then assert the required result. An assertion exposing the seeded bug is useful evidence; a collection error means setup must be repaired first. Rerunning the same expectation checks the correction.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/behavior-test-cycle.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Run the supplied passing checks

The next cell starts a fresh pytest process with this kernel's Python. `-p no:cacheprovider` and `PYTHONDONTWRITEBYTECODE=1` avoid generating test/bytecode caches in the supplied tree. Ordinary manual runs can generate ignored caches. Test discovery is limited to `tests/` in `pyproject.toml`; intentional failures live in `practice/` and require an explicit path.

In [ ]:
import os
import subprocess

test_environment = dict(os.environ, PYTHONDONTWRITEBYTECODE="1")
passing = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
    capture_output=True, text=True, env=test_environment,
)
print(passing.stdout)
assert passing.returncode == 0, passing.stdout + passing.stderr

### See a failure that actually detects the bug

[practice/buggy_mean.py](practice/buggy_mean.py) deliberately divides by one too many and returns zero for empty input. [Its two tests](practice/test_buggy_mean.py) state the correct behavior. The next cell expects a pytest assertion-failure exit code of 1 and two failed tests; it keeps default Run All successful while visibly demonstrating the intended failures.

In [ ]:
broken = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "practice/test_buggy_mean.py"],
    capture_output=True, text=True, env=test_environment,
)
print(broken.stdout)
assert broken.returncode == 1, broken.stdout + broken.stderr
assert "2 failed" in broken.stdout
print("Expected seeded assertion failures observed.")

Read one traceback from the assertion back to the actual value. For `[2, 4]`, the wrong denominator gives `2.0` instead of `3`. Empty input returns `0` instead of `None`. An import error would mean pytest could not reach the behavior; it would not count as detecting these bugs.

Optional illustration: [test-cycle image and attribution](IMAGE_SUGGESTIONS.md). It describes the broader TDD cycle; this exercise only requires a meaningful failing check, a small correction, and another run. Green checks cover the cases you wrote; they are evidence, not proof that every possible input works.

### 🧩 Exercise 03.3.E1 — Expose and correct the mean bug

> **In class · 10 minutes**
>
> **Goal:** Use a requirement to detect a defect rather than making a test match the defect.
>
> **Start here:** The supplied buggy_mean.py and test_buggy_mean.py in the external application copy prepared during the 30-minute setup. If yours is unavailable, use an instructor-prepared scratch copy; do not spend this exercise installing an environment.
>
> **Task:** Write or explain the normal and empty-input tests. Observe their two assertion failures, correct mean only in the scratch copy, and rerun that explicit target. Add a zero-input check and explain why zero and empty have different expected results.
>
> **Check:** The original detects two failures with exit code 1; the corrected scratch copy passes the same tests. Import/collection errors are resolved separately.
>
> **Hint:** The denominator is the number of values. Keep the expected results fixed while correcting the function.

In [ ]:
# Your attempt for 03.3.E1.
# Record code, commands, observations, and explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_03_3_e1.py

## Guided follow-up — Independent work · 15 minutes

Write one independent test for state isolation: adding a record to a left Report must not change a new right Report. Read the supplied `test_report_instances_have_independent_state` for comparison after attempting it. Test externally observable state rather than a private list's exact name. This is a useful regression case for the shared mutable-state mistake from introductory OOP.

Read pytest's summary and note which test failed and why. Rerun a focused target while diagnosing, then run the whole passing suite after fixing the code. If a check still imports an old function in a notebook, restart the kernel; the fresh subprocess checks do not share that import cache.

**Optional extension · 15 minutes, excluded:** deliberately change a copied implementation to return the first value instead of the mean. Predict which existing test will catch that mistake before running it.

## References

- [pytest: first tests](https://docs.pytest.org/en/stable/getting-started.html)
- [Passing tests](tests/test_reports.py)
- [Previous: collaboration](03.2-branches-and-collaboration.ipynb)
- [Next: fixtures and parameters](03.4-fixtures-and-parametrized-tests.ipynb)
- [Class index](README.md)